# Binary pattern search

**Workflow 4 — analyse representations.** Once a score is a binary pitch × time
grid, search is a sliding-window comparison: place a smaller **kernel** on the
matrix, score the overlap, then move the window.

This notebook stays on Bach *Ein feste Burg* and builds three kernels from the
score itself:

1. **Motif** — a short monophonic contour (one voice);
2. **Chord** — a vertical sonority (short time, several pitches);
3. **Texture** — a wider polyphonic patch.

It then explains **normalised overlap** (convolution on binary grids),
**kernel scale factors** (augmentation / diminution of the search window), and
overlays top matches on the piano roll.

Self-contained: it does **not** require [`binary_representations.ipynb`](binary_representations.ipynb)
to have been run, but that notebook is the best introduction to the
MEI ↔ table ↔ binary rotation.

Common-notation MEI only. Companion guide:
[Analyse representations](../docs/guides/analysis.md).

**What you do**

1. Keep the Bach sample URL, or set a local `MEI_SOURCE`.
2. If needed, set `RUN_FETCH = True` once.
3. Build the binary host matrix, inspect the three kernels, run searches, overlay matches.


In [ ]:
# You can leave this cell unchanged.
try:
    from camat import (
        find_camat_root,
        overlay_top_matches_on_piano_roll,
        parse_files,
        resolve_mei_source,
        run_pattern_search,
    )
    from camat.music_utils import create_binary_matrix_bundle, draw_piano_roll
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        find_camat_root,
        overlay_top_matches_on_piano_roll,
        parse_files,
        resolve_mei_source,
        run_pattern_search,
    )
    from camat.music_utils import create_binary_matrix_bundle, draw_piano_roll

import numpy as np
from IPython.display import display


## 1. Parse Bach and build the host binary matrix

Same source and cache rules as the other Workflow 4 notebooks. Notes with
negative onset (pickup) are dropped before gridding so column `0` means time
`0.0`.


In [ ]:
ROOT = find_camat_root()

BACH_SAMPLE_URL = (
    "https://raw.githubusercontent.com/music-encoding/sample-encodings/"
    "main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei"
)

MEI_SOURCE = BACH_SAMPLE_URL
RUN_FETCH = False
PLOTTING_BACKEND = "bokeh"  # "plt" | "bokeh"

# Host-matrix knobs
RESOLUTION_METHOD = "auto"
Y_MODE = "minmax"
ROW_ORDER = "high_to_low"

mei_path = resolve_mei_source(
    MEI_SOURCE,
    fetch=RUN_FETCH,
    shared_cache=True,
    repo_root=ROOT,
)

if mei_path is None:
    print("Skipped. Set RUN_FETCH = True once, or use a local MEI_SOURCE.")
    results = None
    df_pitch = None
    df_pitch_grid = None
    binary = None
else:
    results, dfs_by_name, _last_df = parse_files(
        [str(mei_path)],
        backend="none",
        display_preview_df_pitch=False,
        display_preview_df_events=False,
        include_xml_ids=True,
        parse_enharmonic=True,
        quiet_native_warnings=True,
        use_remote_cache=True,
    )
    df_pitch = results[0]["df_pitch"]
    df_pitch_grid = df_pitch[df_pitch["Global Onset"] >= 0].copy()
    binary = create_binary_matrix_bundle(
        df_pitch_grid,
        source_name="bach_ein_feste_burg",
        results=results,
        resolution_method=RESOLUTION_METHOD,
        y_mode=Y_MODE,
        row_order=ROW_ORDER,
        include_provenance=True,
    )
    host_matrix = binary.matrix
    matrix_meta = binary.meta
    print(f"df_pitch: {len(df_pitch)} rows; grid notes: {len(df_pitch_grid)}")
    print(f"Host matrix: {host_matrix.shape}, resolution={matrix_meta['resolution']}")
    binary.print_summary()
    draw_piano_roll(
        df_pitch_grid,
        measure_offsets=binary.measure_offsets,
        backend=PLOTTING_BACKEND,
        show_measure_lines=True,
        colorize_voices=True,
        show_hover=True,
        plot_width=1000,
        plot_height=360,
    )


## 2. Convolution on a binary grid (foundation of search)

Place kernel $K$ with top-left at host coordinates $(i, j)$. The **raw overlap**
counts shared active cells:

$$
S(i,j)=\sum_{u,v} M[i+u,j+v]\cdot K[u,v]
$$

**Normalised overlap** divides by the number of active kernel cells, so a
perfect fit scores `1.0` regardless of kernel size:

$$
\hat S(i,j)=\frac{S(i,j)}{\sum_{u,v} K[u,v]}
\quad(\text{when the denominator is }>0)
$$

Search slides this window across the host (with optional strides) and can
repeat the scan for **scaled** kernels — stretching or shrinking $K$ in time
(`x`), pitch (`y`), or both. That is how the same motif can match under
augmentation or diminution.

The cell below evaluates one placement so the arithmetic stays visible.


In [ ]:
# Tiny demo kernel: a 2-note rising step on adjacent pitch bins
DEMO_ROW, DEMO_COL = 0, 0

if binary is None:
    print("Skipped. Build the host matrix first.")
else:
    M = np.asarray(binary.matrix, dtype=float)
    K = np.zeros((3, 4), dtype=float)
    K[0, 0:2] = 1  # higher pitch, early
    K[2, 2:4] = 1  # lower pitch, later
    r, c = K.shape
    i, j = DEMO_ROW, DEMO_COL
    window = M[i : i + r, j : j + c]
    product = window * K
    raw = float(product.sum())
    denom = float(K.sum())
    norm = raw / denom if denom else raw
    print("Kernel K:")
    print(K.astype(int))
    print("\nHost window M[i:i+r, j:j+c]:")
    print(window.astype(int))
    print("\nElement-wise product:")
    print(product.astype(int))
    print(f"\nRaw overlap S = {raw:.0f}")
    print(f"Normalised overlap = {norm:.3f}  (S / {denom:.0f})")


## 3. Build three kernels from the score

Kernels are ordinary binary arrays with the **same resolution and pitch axis**
as the host. Here they are cut from Bach rather than drawn freehand:

| Kernel | Idea | Construction |
| --- | --- | --- |
| Motif | melodic contour | `P1 - Voice 1`, first four quarter notes after time 0 |
| Chord | vertical sonority | opening SATB column(s) from the host matrix |
| Texture | local polyphony | a wider opening patch from the host matrix |

Optional freehand kernels can later come from `binary_matrix_designer`
(`camat.binary_matrix_designer`); this tutorial keeps everything reproducible.


In [ ]:
def crop_active_bbox(matrix, pad=0):
    # Trim zero margins so the kernel stays compact.
    mat = np.asarray(matrix, dtype=float)
    if mat.size == 0 or not np.any(mat):
        return mat
    rows = np.where(mat.any(axis=1))[0]
    cols = np.where(mat.any(axis=0))[0]
    r0 = max(0, int(rows.min()) - pad)
    r1 = min(mat.shape[0], int(rows.max()) + 1 + pad)
    c0 = max(0, int(cols.min()) - pad)
    c1 = min(mat.shape[1], int(cols.max()) + 1 + pad)
    return mat[r0:r1, c0:c1].copy()


if binary is None:
    print("Skipped. Build the host matrix first.")
    kernel_motif = kernel_chord = kernel_texture = None
else:
    # Motif: monophonic voice excerpt, rebuilt on the host pitch/time grid
    motif_notes = df_pitch_grid[
        (df_pitch_grid["Voice"] == "P1 - Voice 1")
        & (df_pitch_grid["Global Onset"] >= 0)
        & (df_pitch_grid["Global Onset"] < 4)
    ].copy()
    motif_bundle = create_binary_matrix_bundle(
        motif_notes,
        source_name="motif_v1",
        resolution_method="manual",
        manual_resolution=float(binary.meta["resolution"]),
        y_mode="minmax",
        midi_low=int(binary.meta["y_min"]),
        midi_high=int(binary.meta["y_max"]),
        row_order=str(binary.meta["row_order"]),
        include_provenance=True,
    )
    kernel_motif = crop_active_bbox(motif_bundle.matrix)

    # Chord: short vertical window on the opening sonority
    chord_slice = binary.slice(
        row_start=0,
        col_start=0,
        n_rows=16,
        n_cols=2,
        selection_name="chord",
    )
    kernel_chord = crop_active_bbox(chord_slice.matrix_slice)

    # Texture: wider polyphonic patch (still from the opening)
    texture_slice = binary.slice(
        row_start=0,
        col_start=0,
        n_rows=24,
        n_cols=12,
        selection_name="texture",
    )
    kernel_texture = crop_active_bbox(texture_slice.matrix_slice)

    for label, kernel in [
        ("motif", kernel_motif),
        ("chord", kernel_chord),
        ("texture", kernel_texture),
    ]:
        print(f"{label:8s} shape={kernel.shape} active={int(kernel.sum())}")
        print(kernel.astype(int))
        print()


## 4. Search for the motif (with window-size modulation)

`KERNEL_SCALE_FACTORS` rebuilds the kernel at several widths. With
`KERNEL_SCALE_AXES = ["x"]`, only **time** is stretched or compressed — the
usual rhythmic augmentation / diminution experiment. Set axes to `"y"` or
`"both"` to modulate pitch span as well.

`run_pattern_search(..., backend="none")` computes scores without plotting every
heatmap; we then overlay the top hits on the piano roll.


In [ ]:
# Search knobs
METRICS_TO_RUN = ["normalized_overlap"]
KERNEL_SCALE_FACTORS = [0.75, 1.0, 1.5]  # time compression / identity / augmentation
KERNEL_SCALE_AXES = ["x"]
STRIDE_Y = 1
STRIDE_X = 1
TOP_N_MATCHES = 5

if binary is None or kernel_motif is None:
    print("Skipped. Need host matrix and motif kernel.")
    motif_results = None
else:
    motif_results, motif_kernels, motif_variant, motif_last = run_pattern_search(
        binary.matrix,
        kernel_motif,
        metrics_to_run=METRICS_TO_RUN,
        stride_y=STRIDE_Y,
        stride_x=STRIDE_X,
        kernel_scale_factors=KERNEL_SCALE_FACTORS,
        kernel_scale_axes=KERNEL_SCALE_AXES,
        binarize_scaled_kernel=True,
        binarize_threshold=0.5,
        backend="none",
        plot_scaled_kernels=False,
        top_n_matches=TOP_N_MATCHES,
    )
    print(f"Variants run: {list(motif_results)}")
    print(f"Last variant: {motif_variant}")
    # Prefer the unscaled factor=1.0 variant when present
    variant_key = next(
        (k for k in motif_results if "factor=1" in k or "sx=1.000" in k),
        motif_variant,
    )
    conv_df = motif_results[variant_key]["normalized_overlap"]
    kernel_for_overlay = motif_kernels.get(variant_key, kernel_motif)
    print(
        f"Using variant {variant_key!r} for overlay; "
        f"kernel shape {np.asarray(kernel_for_overlay).shape}"
    )
    plot, df_matches = overlay_top_matches_on_piano_roll(
        df_pitch_grid,
        conv_df,
        kernel_source=kernel_for_overlay,
        meta=binary.meta,
        measure_offsets=binary.measure_offsets,
        top_n=TOP_N_MATCHES,
        plot_width=1000,
        plot_height=400,
        show_kernel=True,
        show=True,
    )
    display(df_matches)


## 5. Search for the chord

A chord kernel is short in time and taller in pitch. Matches should cluster
where the same vertical spacing recurs. Scale factors on `x` matter less here;
the identity scale is usually enough.


In [ ]:
if binary is None or kernel_chord is None:
    print("Skipped. Need host matrix and chord kernel.")
else:
    chord_results, chord_kernels, chord_variant, _ = run_pattern_search(
        binary.matrix,
        kernel_chord,
        metrics_to_run=["normalized_overlap"],
        stride_y=1,
        stride_x=1,
        kernel_scale_factors=[1.0],
        kernel_scale_axes=["x"],
        binarize_scaled_kernel=True,
        backend="none",
        top_n_matches=TOP_N_MATCHES,
    )
    conv_df = chord_results[chord_variant]["normalized_overlap"]
    plot, df_matches = overlay_top_matches_on_piano_roll(
        df_pitch_grid,
        conv_df,
        kernel_source=chord_kernels.get(chord_variant, kernel_chord),
        meta=binary.meta,
        measure_offsets=binary.measure_offsets,
        top_n=TOP_N_MATCHES,
        plot_width=1000,
        plot_height=400,
        show_kernel=True,
        show=True,
    )
    display(df_matches)


## 6. Search for a texture patch

Texture kernels are larger and less unique, so scores spread out and top hits
can be near-duplicates of the extracted patch itself. That contrast is the
point: binary search is only as selective as the kernel.


In [ ]:
if binary is None or kernel_texture is None:
    print("Skipped. Need host matrix and texture kernel.")
else:
    texture_results, texture_kernels, texture_variant, _ = run_pattern_search(
        binary.matrix,
        kernel_texture,
        metrics_to_run=["normalized_overlap"],
        stride_y=1,
        stride_x=1,
        kernel_scale_factors=[1.0],
        kernel_scale_axes=["x"],
        binarize_scaled_kernel=True,
        backend="none",
        top_n_matches=TOP_N_MATCHES,
    )
    conv_df = texture_results[texture_variant]["normalized_overlap"]
    plot, df_matches = overlay_top_matches_on_piano_roll(
        df_pitch_grid,
        conv_df,
        kernel_source=texture_kernels.get(texture_variant, kernel_texture),
        meta=binary.meta,
        measure_offsets=binary.measure_offsets,
        top_n=TOP_N_MATCHES,
        plot_width=1000,
        plot_height=400,
        show_kernel=True,
        show=True,
    )
    display(df_matches)


## 7. Optional: freehand kernels

For exploratory work, `binary_matrix_designer` opens an interactive grid and
exports `binary_matrix_ui`. Pass that array to `run_pattern_search` exactly as
with the programmatic kernels above. Skip this cell in clean-kernel CI runs.


In [ ]:
# Optional interactive designer (leave RUN_DESIGNER = False for non-interactive runs).
RUN_DESIGNER = False

if RUN_DESIGNER:
    from camat.binary_matrix_designer import binary_matrix_designer

    ui = binary_matrix_designer(rows=12, cols=12, flip_vertical=True, display_ui=True)
    print("Draw a pattern, export to binary_matrix_ui, then call run_pattern_search(...).")
else:
    print("Designer skipped (RUN_DESIGNER = False).")


## What was produced?

- A host binary matrix for Bach *Ein feste Burg* (onset ≥ 0 grid).
- Three kernels: motif, chord, texture — cut from the same score.
- A one-placement convolution demo (raw vs normalised overlap).
- Motif search with time-scale modulation; chord and texture searches at identity scale.
- Piano-roll overlays of top normalised-overlap matches.

**Related:** representation rotation in
[`binary_representations.ipynb`](binary_representations.ipynb); DataFrame stats in
[`df_statistics.ipynb`](df_statistics.ipynb). Guide:
[Analyse representations](../docs/guides/analysis.md).
